In [1]:
import pandas as pd

In [2]:
from sklearn.preprocessing import LabelEncoder

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

### Model Building with only skills as parameter

In [5]:
resume_df = pd.read_excel("dataset/resume_dataset.xlsx")
match_df = pd.read_excel("dataset/candidate_role_match.xlsx")

In [6]:
resume_df.head(3)

,Candidate_Name,Education,Skills,Tools,Past_Projects,Experience,Certification
0,shashank,B.Tech,"python, excel, excel",excel,mentioned,mentioned,not mentioned
1,surishetty harivallabha sai,B.Tech,"python, sql, html, css, javascript, photoshop,...","figma, photoshop, git",mentioned,mentioned,mentioned
2,premkumar narayan tambekar,B.Tech,"python, sql, excel, power bi, react, figma, ca...","power bi, figma, canva, excel, git",mentioned,mentioned,mentioned


In [7]:
match_df.head(10)

,Candidate_Name,Role,Match_Percentage,Fit_Category
0,shashank,Data Analytics Intern,40.00,Moderate Fit
1,shashank,Web Development Intern,0.00,Low Fit
2,shashank,Digital Marketing Intern,0.00,Low Fit
3,shashank,Graphic Design Intern,0.00,Low Fit
4,shashank,HR Operations Intern,25.00,Low Fit
5,surishetty harivallabha sai,Data Analytics Intern,40.00,Moderate Fit
6,surishetty harivallabha sai,Web Development Intern,42.86,Moderate Fit
7,surishetty harivallabha sai,Digital Marketing Intern,0.00,Low Fit
8,surishetty harivallabha sai,Graphic Design Intern,50.00,Moderate Fit
9,surishetty harivallabha sai,HR Operations Intern,0.00,Low Fit


In [8]:
best_roles = match_df.sort_values("Match_Percentage", ascending=False)

print(best_roles.shape)
best_roles.head()



(1725, 4)


,Candidate_Name,Role,Match_Percentage,Fit_Category
414,ashik godson britto a s,HR Operations Intern,100.0,Strong Fit
1200,sarat das,Data Analytics Intern,100.0,Strong Fit
1680,w v p s sriraj,Data Analytics Intern,100.0,Strong Fit
1240,riya chavan,Data Analytics Intern,100.0,Strong Fit
1295,sameer ahmed,Data Analytics Intern,100.0,Strong Fit


In [9]:
best_roles = match_df.sort_values("Match_Percentage", ascending=False).drop_duplicates("Candidate_Name")
print(best_roles.shape)   #1725/5

#now we have best role for each percentage

(334, 4)


In [10]:
best_roles.head()

,Candidate_Name,Role,Match_Percentage,Fit_Category
414,ashik godson britto a s,HR Operations Intern,100.0,Strong Fit
1200,sarat das,Data Analytics Intern,100.0,Strong Fit
1680,w v p s sriraj,Data Analytics Intern,100.0,Strong Fit
1240,riya chavan,Data Analytics Intern,100.0,Strong Fit
1295,sameer ahmed,Data Analytics Intern,100.0,Strong Fit


In [11]:
data = pd.merge(resume_df, best_roles[["Candidate_Name","Role"]], on="Candidate_Name")
data.head()

,Candidate_Name,Education,Skills,Tools,Past_Projects,Experience,Certification,Role
0,shashank,B.Tech,"python, excel, excel",excel,mentioned,mentioned,not mentioned,Data Analytics Intern
1,surishetty harivallabha sai,B.Tech,"python, sql, html, css, javascript, photoshop,...","figma, photoshop, git",mentioned,mentioned,mentioned,Graphic Design Intern
2,premkumar narayan tambekar,B.Tech,"python, sql, excel, power bi, react, figma, ca...","power bi, figma, canva, excel, git",mentioned,mentioned,mentioned,Data Analytics Intern
3,candidate name: dasari harshini,B.Tech,python,NaN,mentioned,mentioned,not mentioned,Data Analytics Intern
4,suraj kumar (cid:131) +91 9122601214,NaN,"python, sql, excel, react, node, figma, canva,...","figma, canva, excel, git",mentioned,mentioned,not mentioned,Data Analytics Intern


In [12]:
Candidate_Names=data["Candidate_Name"].copy()

In [13]:
data.isnull().sum()

Candidate_Name      0
Education         194
Skills             34
Tools              96
Past_Projects       0
Experience          0
Certification       0
Role                0
dtype: int64

In [14]:
data=data[["Skills","Role"]]
data.head(2)

,Skills,Role
0,"python, excel, excel",Data Analytics Intern
1,"python, sql, html, css, javascript, photoshop,...",Graphic Design Intern


In [15]:
data.isnull().sum()

Skills    34
Role       0
dtype: int64

In [16]:
data["Skills"] = data["Skills"].fillna("")

In [17]:
data.isnull().sum()

Skills    0
Role      0
dtype: int64

In [18]:
data["Skills"] = data["Skills"].apply(lambda x: [s.strip().lower() for s in str(x).split(",") if s.strip()])  #convert skills to a list


In [19]:
data.head()

,Skills,Role
0,"[python, excel, excel]",Data Analytics Intern
1,"[python, sql, html, css, javascript, photoshop...",Graphic Design Intern
2,"[python, sql, excel, power bi, react, figma, c...",Data Analytics Intern
3,[python],Data Analytics Intern
4,"[python, sql, excel, react, node, figma, canva...",Data Analytics Intern


In [20]:
skills_tools = [
        "python","sql","excel","power bi","tableau",
        "html","css","javascript","react","node",
        "photoshop","figma","illustrator","canva",
        "marketing","seo", "social media marketing",  "content marketing",
        "communication","recruitment", "management","excel"
]


### One-hot encoding for skills

In [21]:

for skill in skills_tools:
    data[skill] = data["Skills"].apply(lambda skills: 1 if skill in skills else 0)

data.head(2)

,Skills,Role,python,sql,excel,power bi,tableau,html,css,javascript,...,figma,illustrator,canva,marketing,seo,social media marketing,content marketing,communication,recruitment,management
0,"[python, excel, excel]",Data Analytics Intern,1,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,"[python, sql, html, css, javascript, photoshop...",Graphic Design Intern,1,1,0,0,0,1,1,1,...,1,0,0,0,0,0,0,0,0,0


In [22]:
# Droping original Skills column
data = data.drop(columns=["Skills"])
data.head()

,Role,python,sql,excel,power bi,tableau,html,css,javascript,react,...,figma,illustrator,canva,marketing,seo,social media marketing,content marketing,communication,recruitment,management
0,Data Analytics Intern,1,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,Graphic Design Intern,1,1,0,0,0,1,1,1,0,...,1,0,0,0,0,0,0,0,0,0
2,Data Analytics Intern,1,1,1,1,0,0,0,0,1,...,1,0,1,0,0,0,0,1,0,1
3,Data Analytics Intern,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,Data Analytics Intern,1,1,1,0,0,0,0,0,1,...,1,0,1,1,0,0,0,0,0,1


### label encoding for target role column

In [23]:
encoder = LabelEncoder()


data["Role_Encoded"] = encoder.fit_transform(data["Role"])

data=data.drop(columns=["Role"])

data.head(2)

,python,sql,excel,power bi,tableau,html,css,javascript,react,node,...,illustrator,canva,marketing,seo,social media marketing,content marketing,communication,recruitment,management,Role_Encoded
0,1,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,1,0,0,0,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,2


### Encoder Mapping

In [24]:
encoder.classes_

array(['Data Analytics Intern', 'Digital Marketing Intern',
       'Graphic Design Intern', 'HR Operations Intern',
       'Web Development Intern'], dtype=object)

In [25]:
role_label_df = pd.DataFrame({
    "Role": encoder.classes_,
    "Label": encoder.transform(encoder.classes_)
})
role_label_df

,Role,Label
0,Data Analytics Intern,0
1,Digital Marketing Intern,1
2,Graphic Design Intern,2
3,HR Operations Intern,3
4,Web Development Intern,4


### Splitting data in train_test for model training

In [26]:

Y=data["Role_Encoded"]
X=data.drop(columns=["Role_Encoded"])

In [27]:
X.head(1)

,python,sql,excel,power bi,tableau,html,css,javascript,react,node,...,figma,illustrator,canva,marketing,seo,social media marketing,content marketing,communication,recruitment,management
0,1,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [28]:
Y.head()

0    0
1    2
2    0
3    0
4    0
Name: Role_Encoded, dtype: int64

In [29]:

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)


In [30]:
X_train.shape

(276, 21)

In [31]:
X_test.shape

(69, 21)

In [32]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier()
}

# Track best model
best_model_name = None
best_model = None
best_accuracy = 0.0

# Train and evaluate
for name, model in models.items():
    model.fit(X_train, y_train)

    # Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Accuracy
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_test_pred)

    print(f"\n{name}")
    print(f"Train Accuracy: {train_acc:.2f}")
    print(f"Test Accuracy: {test_acc:.2f}")
    print("Confusion Matrix:\n", cm)

    # Update best model if this one is better
    if test_acc > best_accuracy:
        best_accuracy = test_acc
        best_model_name = name
        best_model = model

print("\nBest Model:", best_model_name)
print("Best Test Accuracy:", best_accuracy)


Logistic Regression
Train Accuracy: 0.93
Test Accuracy: 0.87
Confusion Matrix:
 [[16  0  0  0  1]
 [ 0  0  0  2  0]
 [ 0  0  1  1  1]
 [ 2  0  1 22  0]
 [ 0  0  0  1 21]]

Decision Tree
Train Accuracy: 0.96
Test Accuracy: 0.80
Confusion Matrix:
 [[14  0  0  1  2]
 [ 0  0  1  1  0]
 [ 0  0  1  1  1]
 [ 1  1  1 21  1]
 [ 1  0  0  2 19]]

Random Forest
Train Accuracy: 0.96
Test Accuracy: 0.87
Confusion Matrix:
 [[16  0  0  0  1]
 [ 0  0  1  1  0]
 [ 0  0  1  1  1]
 [ 3  0  0 22  0]
 [ 0  0  0  1 21]]

Best Model: Logistic Regression
Best Test Accuracy: 0.8695652173913043


#### Top-3 role predictions

In [35]:
prob_matrix = best_model.predict_proba(X)
roles = encoder.classes_   # map back to role names

top_roles = []
for probs in prob_matrix:
    role_scores = list(zip(roles, probs*100))
    role_scores.sort(key=lambda x: x[1], reverse=True)
    top3 = role_scores[:3]
    top_roles.append(top3)

# Build results DataFrame with candidate names
results = []
for i, candidate in enumerate(Candidate_Names):
    r1, r2, r3 = top_roles[i]
    results.append({
        "Candidate_Name": candidate,
        "Top_Role_1": r1[0], "Score_1": round(r1[1], 2),
        "Top_Role_2": r2[0], "Score_2": round(r2[1], 2),
        "Top_Role_3": r3[0], "Score_3": round(r3[1], 2)
    })

result_df = pd.DataFrame(results)

# Save results
result_df.to_excel("dataset/top3_role_predictions_structured.xlsx", index=False)

print("\nTop 3 role predictions saved successfully!")


Top 3 role predictions saved successfully!


### Saving models

In [37]:
import joblib
import os

output_folder = "Models"
os.makedirs(output_folder, exist_ok=True)

# Save all models
for name, model in models.items():
    filename = f"{output_folder}/{name.replace(' ', '_').lower()}_model.pkl"
    joblib.dump(model, filename)
    print(f"Saved {name} as {filename}")

# Save the best model separately
best_filename = f"{output_folder}/best_model_{best_model_name.replace(' ', '_').lower()}.pkl"
joblib.dump(best_model, best_filename)
print(f"\nBest model ({best_model_name}) saved as {best_filename}")

Saved Logistic Regression as logistic_regression_model.pkl
Saved Decision Tree as decision_tree_model.pkl
Saved Random Forest as random_forest_model.pkl

Best model (Logistic Regression) saved as best_model_Logistic Regression.pkl


# Advanced  :With Certifications,projects,experience(Incomplete)

In [ ]:
data["Past_Projects"]=data["Past_Projects"].fillna("not mentioned")
data["Education"] = data["Education"].fillna("not mentioned")
data["Tools"] = data["Tools"].fillna("not mentioned")


In [ ]:
data["Past_Projects"] = data["Past_Projects"].apply(lambda x: 0 if x=="not mentioned" else 1)
data["Education"] = data["Education"].apply(lambda x: 0 if x=="not mentioned" else 1)
data["Tools"] = data["Tools"].apply(lambda x: 0 if x=="not mentioned" else 1)
data["Experience"] = data["Experience"].apply(lambda x: 0 if x=="not mentioned" else 1)
data["Certification"] = data["Certification"].apply(lambda x: 0 if x=="not mentioned" else 1)

In [ ]:
data.head()

In [ ]:
data.isnull().sum()

In [ ]:
# Convert to lowercase and split by comma
all_skills = data["Skills"].dropna().str.lower().str.split(",")

# Flatten the list of lists
flat_skills = [skill.strip() for sublist in all_skills for skill in sublist]

# Get unique skills
unique_skills = sorted(set(flat_skills))
print("Total unique skills:", len(unique_skills))
print(unique_skills[:20])  # preview first 20


In [ ]:
X_text.head()